In [0]:
%sql

-- Create catalog for the entire project
CREATE CATALOG IF NOT EXISTS ecommerce_pipeline
COMMENT 'Analytics Engineering Portfolio Project';

-- Create schemas for each pipeline layer  
CREATE SCHEMA IF NOT EXISTS ecommerce_pipeline.raw
COMMENT 'Raw data as ingested from source';

CREATE SCHEMA IF NOT EXISTS ecommerce_pipeline.staging
COMMENT 'Cleaned and typed data';

CREATE SCHEMA IF NOT EXISTS ecommerce_pipeline.intermediate
COMMENT 'Business logic applied';

CREATE SCHEMA IF NOT EXISTS ecommerce_pipeline.mart
COMMENT 'Final tables for reporting';

In [0]:
%sql
SHOW SCHEMAS IN ecommerce_pipeline;

databaseName
default
information_schema
intermediate
mart
raw
staging


In [0]:
%sql

-- ============================================================
-- Load raw October 2019 events into Delta table
-- Source: CSV file in Volume (landing zone)
-- Target: ecommerce_pipeline.raw.events_oct
-- Method: CREATE TABLE AS SELECT (CTAS)
-- ============================================================

CREATE TABLE IF NOT EXISTS ecommerce_pipeline.raw.events_oct
COMMENT 'Raw eCommerce behavioral events - October 2019. Source: Kaggle mkechinov dataset.'
AS 
SELECT * 
FROM read_files(
    '/Volumes/workspace/default/ecommerce_raw/2019-Oct.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) FROM ecommerce_pipeline.raw.events_oct;

COUNT(*)
42448764


In [0]:
%sql

-- ============================================================
-- Load raw November 2019 events into Delta table
-- Source: CSV file in Volume (landing zone)
-- Target: ecommerce_pipeline.raw.events_nov
-- Method: CREATE TABLE AS SELECT (CTAS)
-- ============================================================

CREATE TABLE IF NOT EXISTS ecommerce_pipeline.raw.events_nov
COMMENT 'Raw eCommerce behavioral events - November 2019. Source: Kaggle mkechinov dataset.'
AS 
SELECT * 
FROM read_files(
    '/Volumes/workspace/default/ecommerce_raw/2019-Nov.csv',
    format => 'csv',
    header => true,
    inferSchema => true
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) FROM ecommerce_pipeline.raw.events_nov;

COUNT(*)
67501979


In [0]:
%sql

-- Check table schema and data types
DESCRIBE TABLE ecommerce_pipeline.raw.events_oct;

col_name,data_type,comment
event_time,timestamp,null
event_type,string,null
product_id,int,null
category_id,bigint,null
category_code,string,null
brand,string,null
price,double,null
user_id,int,null
user_session,string,null
_rescued_data,string,null


In [0]:
%sql

-- Check if any data was rescued (failed to parse)
SELECT COUNT(*) as rescued_count
FROM ecommerce_pipeline.raw.events_oct
WHERE _rescued_data IS NOT NULL;

rescued_count
0


In [0]:
%sql

-- Preview first 5 rows
SELECT * EXCEPT(_rescued_data)
FROM ecommerce_pipeline.raw.events_oct
LIMIT 5;

event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
2019-10-03T12:26:50.000Z,view,17900196,2053013560178508465,construction.tools.generator,firman,1377.25,514465215,a91dceb0-3439-4774-ac99-6bb441a392f5
2019-10-03T12:26:50.000Z,view,1002524,2053013555631882655,electronics.smartphone,apple,513.4,503347063,971c9398-ff5e-4cbb-8f20-7c007c696875
2019-10-03T12:26:50.000Z,view,1004856,2053013555631882655,electronics.smartphone,samsung,132.17,523588323,54bf6c25-54db-49d2-bfa1-e3a0124e9354
2019-10-03T12:26:50.000Z,view,3200013,2053013555321504139,appliances.kitchen.meat_grinder,panasonic,101.65,515045171,ad9e7339-2d3b-45d4-a4bd-ec014ca5f2cd
2019-10-03T12:26:50.000Z,view,1005135,2053013555631882655,electronics.smartphone,apple,1747.79,514904179,6c758cd3-84d8-4ee3-a31a-559ae7301e48


In [0]:
%sql

-- Show all tables in raw schema
SHOW TABLES IN ecommerce_pipeline.raw;

database,tableName,isTemporary
raw,events_nov,false
raw,events_oct,false


In [0]:
%sql

-- Rename tables to include year for clarity
ALTER TABLE ecommerce_pipeline.raw.events_oct RENAME TO ecommerce_pipeline.raw.events_oct_2019;
ALTER TABLE ecommerce_pipeline.raw.events_nov RENAME TO ecommerce_pipeline.raw.events_nov_2019;

In [0]:
%sql
SHOW TABLES IN ecommerce_pipeline.raw;

database,tableName,isTemporary
raw,events_nov_2019,false
raw,events_oct_2019,false
